# Python Code Development for HCMST Analysis

In [1]:
import diversedata as dd
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import altair as alt
from IPython.display import Markdown

## Data Cleaning and Processing

In [2]:
hcmst = dd.load_data("hcmst")

# Review total rows
hcmst.shape[0]

# Select features of interest and removing NA
hcmst = hcmst[
    [
        "same_sex_couple",
        "sex_frequency",
        "flirts_with_partner",
        "fights_with_partner",
        "inc_change_during_pandemic",
        "subject_had_covid",
        "partner_had_covid",
        "subject_vaccinated",
        "partner_vaccinated",
        "agree_covid_approach",
        "relationship_quality",
    ]
].dropna()

# Remaining row count.
hcmst.shape[0]

1220

## 2. Variable Encoding

In [3]:
# check R level prints

In [4]:
hcmst['same_sex_couple'] = pd.Categorical(
    hcmst['same_sex_couple'].replace({'no': 'No', 'yes': 'Yes'}),
    categories=['No', 'Yes'],
)
print(f'same_sex_couple levels: {hcmst["same_sex_couple"].cat.categories.tolist()}')

same_sex_couple levels: ['No', 'Yes']


In [5]:
behavioral_variables_levels = {
    "sex_frequency" : [
        "once_a_month_or_less", "2_to_3_times_a_month",
        "once_or_twice_a_week", "3_to_6_times_a_week",
        "once_or_more_a_day"
    ],
    "flirts_with_partner" : [
        "never", "less_than_once_a_month", 
        "1_to_3_times_a_month", "once_a_week", 
        "a_few_times_a_week", "every_day"
    ],
    "fights_with_partner" : [
        "0_times", "1_time", "2_times", "3_times", "4_times", 
        "5_times", "6_times", "7_or_more_times"
    ],
}

for variable, levels in behavioral_variables_levels.items():
    hcmst[variable] = pd.Categorical(
        hcmst[variable],
        categories=levels,
        ordered=True
    )
    print(f'{variable} levels: {", ".join(hcmst[variable].cat.categories)} \n')

sex_frequency levels: once_a_month_or_less, 2_to_3_times_a_month, once_or_twice_a_week, 3_to_6_times_a_week, once_or_more_a_day 

flirts_with_partner levels: never, less_than_once_a_month, 1_to_3_times_a_month, once_a_week, a_few_times_a_week, every_day 

fights_with_partner levels: 0_times, 1_time, 2_times, 3_times, 4_times, 5_times, 6_times, 7_or_more_times 



In [6]:
covid_19_variables_levels = {
    "inc_change_during_pandemic": [
        "much_worse", "worse", "no_change", "better", "much_better"
    ],
    "subject_had_covid": [
        "no", "yes"
    ],
    "partner_had_covid": [
        "no", "yes"
    ],
    "subject_vaccinated": [
        "not_vaccinated", "partially_vaccinated", 
        "fully_vaccinated_no_booster", "fully_vaccinated_and_booster"
    ],
    "partner_vaccinated": [
        "not_vaccinated", "partially_vaccinated", 
        "fully_vaccinated_no_booster", "fully_vaccinated_and_booster"
    ],
    "agree_covid_approach": [
        "completely_disagree", "mostly_disagree", 
        "mostly_agree", "completely_agree"
    ],
}

for variable, levels in covid_19_variables_levels.items():
    hcmst[variable] = pd.Categorical(
        hcmst[variable],
        categories=levels,
        ordered=True
    )
    print(f'{variable} levels: {", ".join(hcmst[variable].cat.categories)} \n')

inc_change_during_pandemic levels: much_worse, worse, no_change, better, much_better 

subject_had_covid levels: no, yes 

partner_had_covid levels: no, yes 

subject_vaccinated levels: not_vaccinated, partially_vaccinated, fully_vaccinated_no_booster, fully_vaccinated_and_booster 

partner_vaccinated levels: not_vaccinated, partially_vaccinated, fully_vaccinated_no_booster, fully_vaccinated_and_booster 

agree_covid_approach levels: completely_disagree, mostly_disagree, mostly_agree, completely_agree 



In [7]:
hcmst["relationship_quality"] = pd.Categorical(
    hcmst["relationship_quality"],
    categories=[
        "very_poor", "poor", "fair", "good", "excellent"
    ],
    ordered=True,
)
print(f'relationship_quality levels: {hcmst["relationship_quality"].cat.categories.tolist()}')

## change R print to relationship_quality instead of sex freq

relationship_quality levels: ['very_poor', 'poor', 'fair', 'good', 'excellent']


## 3. Exploratory Data Analysis

In [8]:
alt.Chart(hcmst).mark_bar().encode(
    x=alt.X("same_sex_couple").title("Same Sex Couple"),
    y=alt.Y("count()").title("Count"),
    color=alt.Color("same_sex_couple", legend=None),
).properties(title="Couple Type Distribution", width=350, height=350)

alt.Chart(...)

In [9]:
def histogram_plot(data, y_var, facet, label_map, plot_title="Relative Frequency by Category"):
    data_cleaned = data.groupby(by=[y_var, facet], observed=False).size().reset_index(name='count')
    data_cleaned['prop'] = data_cleaned.groupby(facet, observed=False)['count'].transform(lambda x: x / x.sum())
    data_cleaned[y_var] = data_cleaned[y_var].cat.rename_categories(label_map)

    plot = alt.Chart(data_cleaned).mark_bar(size=10).encode(
        x=alt.X('prop').title('Relative Frequency'),
        y=alt.Y(y_var).title(None).scale(alt.Scale(reverse=True)),
        yOffset='same_sex_couple',
        color=alt.Color('same_sex_couple').title("Same Sex Couple"),
    ).properties(title=plot_title, width=400, height=alt.Step(10))
    
    return plot

In [10]:
sex_freq_labels = {
  "once_a_month_or_less": "Once a Month or Less",
  "2_to_3_times_a_month": "Two or Three Times a Month",
  "once_or_twice_a_week": "Once or Twice a Week",
  "3_to_6_times_a_week": "Three to Six Times a Week",
  "once_or_more_a_day": "Once or More a Day"
}
hist_sex_freq = histogram_plot(hcmst, 'sex_frequency', 'same_sex_couple', sex_freq_labels, 'Sex Frequency')

flirt_freq_labels = {
  "never": "Never",
  "less_than_once_a_month": "Less Than Once a Month",
  "1_to_3_times_a_month": "One to Three Times a Month",
  "once_a_week": "Once a Week",
  "a_few_times_a_week": "A Few Times a Week",
  "every_day": "Every Day" 
}
hist_flirt_freq = histogram_plot(hcmst, "flirts_with_partner", "same_sex_couple", flirt_freq_labels, "Flirting Frequency")

fight_freq_labels = {
  "0_times": "Zero Times",
  "1_time": "One Time",
  "2_times": "Two Times",
  "3_times": "Three Times",
  "4_times": "Four Times",
  "5_times": "Five Times",
  "6_times": "Six Times",
  "7_or_more_times": "Seven or More Times"
}
hist_fight_freq = histogram_plot(hcmst, "fights_with_partner", "same_sex_couple", fight_freq_labels, "Fighting Frequency")

inc_labels = {
  "much_worse": "Much Worse",
  "worse": "Worse",
  "no_change": "No Change",
  "better": "Better",
  "much_better": "Much Better"
}
hist_inc_change_freq = histogram_plot(hcmst, "inc_change_during_pandemic", "same_sex_couple", inc_labels, "Income Change During Pandemic Frequency")

yes_no_label = {
    "no": "No", "yes": "Yes"
}
hist_sub_covid_freq = histogram_plot(hcmst, "subject_had_covid", "same_sex_couple", yes_no_label, "Subject had COVID-19 Frequency")
hist_par_covid_freq = histogram_plot(hcmst, "partner_had_covid", "same_sex_couple", yes_no_label, "Partner had COVID-19 Frequency")

vax_label ={
  "not_vaccinated": "Not Vaccinated",
  "partially_vaccinated": "Partially Vaccinated",
  "fully_vaccinated_no_booster": "Fully Vaccinated No Booster",
  "fully_vaccinated_and_booster": "Fully Vaccinated and Booster"
}
hist_sub_vax_freq = histogram_plot(hcmst, "subject_vaccinated", "same_sex_couple", vax_label, "Subject Vaccination Status Frequency")
hist_par_vax_freq = histogram_plot(hcmst, "partner_vaccinated", "same_sex_couple", vax_label, "Partner Vaccination Status Frequency")

approach_label = {
  "completely_disagree": "Completely Disagree",
  "mostly_disagree": "Mostly Disagree",
  "mostly_agree": "Mostly Agree",
  "completely_agree": "Completely Agree"
}
hist_cov_approach_freq = histogram_plot(hcmst, "agree_covid_approach", "same_sex_couple", approach_label, "Agreement on Pandemic Approach Frequency")

quality_label = {
  "very_poor": "Very Poor",
  "poor": "Poor",
  "fair": "Fair",
  "good": "Good",
  "excellent": "Excellent"
}
hist_quality_freq = histogram_plot(hcmst, "relationship_quality", "same_sex_couple", quality_label, "Relationship Quality Frequency")

alt.vconcat(
  hist_sex_freq,
  hist_flirt_freq,
  hist_fight_freq,
  hist_inc_change_freq,
  hist_sub_covid_freq,
  hist_par_covid_freq,
  hist_sub_vax_freq,
  hist_par_vax_freq,
  hist_cov_approach_freq,
  hist_quality_freq
).resolve_legend(color='independent')

alt.VConcatChart(...)

## 4. Model Fitting and Assumptions

In [11]:
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

exog = hcmst[[
    'same_sex_couple',
    'sex_frequency',
    'flirts_with_partner',
    'fights_with_partner',
    'inc_change_during_pandemic',
    'subject_had_covid',
    'partner_had_covid',
    'subject_vaccinated',
    'partner_vaccinated',
    'agree_covid_approach'
]]

# If your variables are categorical/ordinal, encode them
exog = pd.get_dummies(exog, drop_first=True)

# Fit the ordinal logistic model
model = OrderedModel(
    endog=hcmst['relationship_quality'],
    exog=exog,
    distr='logit'  # matches polr() which uses logit by default
)

result = model.fit(method='bfgs')  # equivalent to using Hess=TRUE in polr
result

Optimization terminated successfully.
         Current function value: 0.879911
         Iterations: 169
         Function evaluations: 170
         Gradient evaluations: 170


In [13]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                              OrderedModel Results                              
================================================================================
Dep. Variable:     relationship_quality   Log-Likelihood:                -1073.5
Model:                     OrderedModel   AIC:                             2219.
Method:              Maximum Likelihood   BIC:                             2403.
Date:                  Wed, 06 Aug 2025                                         
Time:                          20:49:49                                         
No. Observations:                  1220                                         
Df Residuals:                      1184                                         
Df Model:                            32                                         
===================================================================================================================
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
same_sex_couple_Yes                                -0.5296      0.220     -2.411      0.016      -0.960      -0.099
sex_frequency_2_to_3_times_a_month                  0.6415      0.154      4.176      0.000       0.340       0.943
sex_frequency_once_or_twice_a_week                  0.8070      0.160      5.049      0.000       0.494       1.120
sex_frequency_3_to_6_times_a_week                   1.0559      0.252      4.198      0.000       0.563       1.549
sex_frequency_once_or_more_a_day                    1.2125      0.567      2.140      0.032       0.102       2.323
flirts_with_partner_less_than_once_a_month          0.0669      0.170      0.394      0.693      -0.266       0.400
flirts_with_partner_1_to_3_times_a_month            0.5188      0.247      2.096      0.036       0.034       1.004
flirts_with_partner_once_a_week                    -0.0970      0.252     -0.385      0.700      -0.591       0.397
flirts_with_partner_a_few_times_a_week              0.3065      0.176      1.739      0.082      -0.039       0.652
flirts_with_partner_every_day                       1.1949      0.259      4.607      0.000       0.687       1.703
fights_with_partner_1_time                         -0.6506      0.143     -4.552      0.000      -0.931      -0.371
fights_with_partner_2_times                        -0.9257      0.200     -4.625      0.000      -1.318      -0.533
fights_with_partner_3_times                        -1.4801      0.290     -5.100      0.000      -2.049      -0.911
fights_with_partner_4_times                        -2.3293      0.445     -5.234      0.000      -3.202      -1.457
fights_with_partner_5_times                        -3.5806      0.603     -5.937      0.000      -4.763      -2.399
fights_with_partner_6_times                        -2.2828      0.620     -3.682      0.000      -3.498      -1.068
fights_with_partner_7_or_more_times                -2.9423      0.553     -5.323      0.000      -4.026      -1.859
inc_change_during_pandemic_worse                    0.3436      0.296      1.160      0.246      -0.237       0.924
inc_change_during_pandemic_no_change                0.8735      0.269      3.246      0.001       0.346       1.401
inc_change_during_pandemic_better                   1.0225      0.319      3.210      0.001       0.398       1.647
inc_change_during_pandemic_much_better              1.0843      0.542      2.002      0.045       0.023       2.146
subject_had_covid_yes                              -0.0741      0.173     -0.429      0.668      -0.413       0.265
partner_had_covid_yes                               0.1165      0.173      0.675      0.499      -0.222       0.455
subject_vaccinated_partially_vaccinated            -0.2622      0.718     -0.365      0.715      -1.669       1.145
subject_va

In [ ]:
# doesn't seem like there is an easy way to do the brant test, also can't do successive contrasts
# prob better suited to do a 